# Read JSON to MinIO Iceberg

This notebook reads `sample.json` containing BusWayPoint data and writes it into an Iceberg table on MinIO.
It is based on the logic from `read_json_to_minio.py`.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    FloatType, DoubleType, BooleanType
)
from pyspark.sql.functions import current_timestamp, from_unixtime, to_timestamp, to_date

### Initialize Spark Session
Configure Spark to connect to the Iceberg REST Catalog and MinIO.

In [ ]:
spark = SparkSession.builder \
    .appName("ReadJsonToMinIO-Notebook") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "rest") \
    .config("spark.sql.catalog.iceberg.uri", "http://gravitino:9001/api/metalakes/default/catalogs/iceberg") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://lakehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session created successfully!")

### Define schema based on `ufms.proto`

In [ ]:
bus_way_point_schema = StructType([
    StructField("vehicle", StringType(), True),
    StructField("driver", StringType(), True),
    StructField("speed", FloatType(), True),
    StructField("datetime", IntegerType(), True),
    StructField("x", DoubleType(), True),
    StructField("y", DoubleType(), True),
    StructField("z", FloatType(), True),
    StructField("heading", FloatType(), True),
    StructField("ignition", BooleanType(), True),
    StructField("aircon", BooleanType(), True),
    StructField("door_up", BooleanType(), True),
    StructField("door_down", BooleanType(), True),
    StructField("sos", BooleanType(), True),
    StructField("working", BooleanType(), True),
    StructField("analog1", FloatType(), True),
    StructField("analog2", FloatType(), True)
])

root_schema = StructType([
    StructField("msgType", StringType(), True),
    StructField("msgBusWayPoint", bus_way_point_schema, True)
])

### Load Data

In [ ]:
json_path = "/opt/spark/apps/data/HPCLAB/sample.json"

# Read the JSON file (it's an array of objects, so multiline=True)
df = spark.read.schema(root_schema).option("multiline", "true").json(json_path)
print(f"Loaded {df.count()} records from JSON.")
df.show(5, truncate=False)

### Transform and Flatten Data
Flattens the nested structs and extracts the date for partitioning.

In [ ]:
df_flattened = df.select("msgType", "msgBusWayPoint.*")

# Convert epoch datetime to Spark timestamp and extract Date for partitioning
df_with_ts = (
    df_flattened
    .withColumn("timestamp", to_timestamp(from_unixtime("datetime")))
    .withColumn("date", to_date(from_unixtime("datetime")))
    .withColumn("load_at", current_timestamp())
)

print("Transformed DataFrame Schema:")
df_with_ts.printSchema()

print("Sample Data:")
df_with_ts.select("vehicle", "datetime", "timestamp", "date").show(5, truncate=False)

### Write to MinIO Iceberg
Ensure that the namespace / schema exists before writing the table.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.bus_bronze")

In [ ]:
table_name = "iceberg.bus_bronze.bus_way_points"

print(f"Writing data to {table_name} partitioned by 'date'...")

(
    df_with_ts
    .write.format("iceberg")
    .mode("overwrite")
    .partitionBy("date")
    .saveAsTable(table_name)
)

print("Write completed successfully!")

### Verify Written Data

In [ ]:
verify_df = spark.read.table(table_name)
print(f"Record count in {table_name}: {verify_df.count()}")
verify_df.show(5)